# Tübingen Bicycle Counter Analysis

This notebook demonstrates the new modular tuecycle package for analyzing bike counter data with weather correlations.

## Quick Start

```python
from tuecycle import DataManager
from tuecycle.plots import get_plot, list_plots

dm = DataManager()
df = dm.get("tuebingen_tunnel")
fig = get_plot("hourly_pattern")(df, "Tübingen Radtunnel")
fig.show()
```

## 1. Setup and Imports

First, let's import the tuecycle package and see what's available.

In [11]:
# Import the tuecycle package
from tuecycle import DataManager, get_station
from tuecycle.config import list_stations_with_weather
from tuecycle.plots import get_plot, list_plots

# Show available stations
print("Available stations:")
for alias in list_stations_with_weather():
    station = get_station(alias)
    print(f"  • {alias}: {station.display_name}")

Available stations:
  • freiburg_dreisam: Freiburg (Dreisam)
  • freiburg_eschholz: Freiburg (Eschholzstraße)
  • freiburg_gueterbahn: Freiburg (Güterbahn)
  • freiburg_wiwili: Freiburg (Wiwilibrücke)
  • heidelberg_ernst_walz: Heidelberg (Ernst-Walz-Brücke)
  • heidelberg_gaisberg: Heidelberg (Gaisbergstraße)
  • heidelberg_kurfuersten: Heidelberg (Kurfürstenanlage)
  • heidelberg_liebermann: Heidelberg (Liebermannstraße)
  • heidelberg_mannheimer: Heidelberg (Mannheimer Straße)
  • heidelberg_ploeck: Heidelberg (Plöck)
  • heidelberg_rohrbacher: Heidelberg (Rohrbacher Straße)
  • heidelberg_schlierbacher: Heidelberg (Schlierbacher Landstraße)
  • heidelberg_theodor_heuss: Heidelberg (Theodor-Heuss-Brücke)
  • heidelberg_ziegelhaeuser: Heidelberg (Ziegelhäuser Landstraße)
  • heilbronn_neckarufer: Heilbronn (Neckarufer)
  • heilbronn_nord: Heilbronn (Route Nord)
  • heilbronn_sued: Heilbronn (Route Süd)
  • karlsruhe_erbprinzen: Karlsruhe (Erbprinzenstraße)
  • kirchheim_barometer: Ki

In [12]:
# Show available plots
print("\nAvailable plots:")
for name in list_plots():
    print(f"  • {name}")


Available plots:
  • {'name': 'bike_vs_rain_rush_hour_city', 'description': 'Bike counter with rain at ≤5°C - Rush Hour'}
  • {'name': 'bike_vs_temp_heatmap', 'description': '2D density heatmap of bike counts vs temperature'}
  • {'name': 'city_comparison_hourly', 'description': 'Compare hourly patterns across stations'}
  • {'name': 'city_comparison_monthly', 'description': 'Compare monthly patterns across stations (normalized)'}
  • {'name': 'city_rain_share_boxplot', 'description': 'Boxplot showing the share of bike rides during rain per city (Rush Hour)'}
  • {'name': 'deviation_from_baseline', 'description': 'Bike count deviation from rolling baseline by rain category'}
  • {'name': 'fourier_transform', 'description': 'Fourier transform showing periodic patterns'}
  • {'name': 'hour_weekday_heatmap', 'description': 'Heatmap of bike counts by hour and day of week'}
  • {'name': 'hourly_pattern', 'description': 'Average bike count by hour of day with std band'}
  • {'name': 'monthl

In [13]:
PLOT_STATIONS = ["tuebingen_tunnel", "heidelberg_kurfuersten", "stuttgart_koenig_karls"]

START_DATE = (2024, 11, 1)
END_DATE = (2025, 10, 31)

## 2. Load Data with Parquet Caching

The `DataManager` handles loading bike counter data merged with weather data. On first run, it loads from CSVs and caches to Parquet. Subsequent loads are **10-50x faster**.

In [14]:
import time

# Initialize the DataManager
dm = DataManager(start_date=START_DATE,
                 end_date=END_DATE)
# Load all comparison stations at once using the configuration
start = time.time()
dfs = dm.get_multiple(PLOT_STATIONS)
print(f"Loaded {len(PLOT_STATIONS)} stations in {time.time() - start:.2f}s")

Loaded 3 stations in 0.01s


In [15]:
# All stations are already loaded in dfs!
# Print summary of all loaded stations
print("Loaded stations:")
for alias, df in dfs.items():
    station = get_station(alias)
    print(f"  • {station.display_name}: {len(df):,} rows")

Loaded stations:
  • Tübingen (Radtunnel): 8,760 rows
  • Heidelberg (Kurfürstenanlage): 8,760 rows
  • Stuttgart (König-Karls-Brücke): 8,760 rows


## 3. Single-Station Plots

### Time Series Overview
Interactive plot showing bike counts, temperature, and rainfall with range slider.

In [16]:
# Time series plot
for alias in PLOT_STATIONS:
    df = dfs[alias]
    station = get_station(alias)
    fig = get_plot("time_series")(df, title=station.display_name)
    fig.show()

### Hourly Pattern
Average bike count for each hour of the day with ±1 standard deviation band.

In [17]:
for alias in PLOT_STATIONS:
    df = dfs[alias]
    station = get_station(alias)
    fig = get_plot("hourly_pattern")(df, title=station.display_name)
    fig.show()

### Seasonal Comparison
Compare hourly patterns across:
- **Winter:** Nov, Dec, Jan, Feb
- **Transition:** Mar, Apr, Sep, Oct  
- **Summer:** May, Jun, Jul, Aug

In [18]:
for alias in PLOT_STATIONS:
    df = dfs[alias]
    station = get_station(alias)
    fig = get_plot("seasonal_comparison")(df, title=station.display_name)
    fig.show()

### Weekday vs Weekend

In [19]:
for alias in PLOT_STATIONS:
    df = dfs[alias]
    station = get_station(alias)
    fig = get_plot("weekday_vs_weekend")(df, title=station.display_name)
    fig.show()

### Hour × Weekday Heatmap

In [20]:
for alias in PLOT_STATIONS:
    df = dfs[alias]
    station = get_station(alias)
    fig = get_plot("hour_weekday_heatmap")(df, title=station.display_name)
    fig.show()

### Monthly Average

In [21]:
for alias in PLOT_STATIONS:
    df = dfs[alias]
    station = get_station(alias)
    fig = get_plot("monthly_average")(df, title=station.display_name)
    fig.show()

### Bike vs Temperature Heatmap
2D density showing relationship between temperature and bike counts (daytime hours only).

In [22]:
for alias in PLOT_STATIONS:
    df = dfs[alias]
    station = get_station(alias)
    fig = get_plot("bike_vs_temp_heatmap")(df, title=station.display_name)
    fig.show()

### Temperature Deviation Heatmap
Removes seasonal bias by showing how temperature deviation from monthly average affects bike counts.
- **Slope** quantifies sensitivity: % change in bike count per °C deviation

In [23]:
for alias in PLOT_STATIONS:
    df = dfs[alias]
    station = get_station(alias)
    fig = get_plot("temp_deviation_heatmap")(df, title=station.display_name)
    fig.show()

### Rush Hour Analysis
Compare average bike counts across time categories:
- Morning Rush (7-9 AM weekdays)
- Evening Rush (5-7 PM weekdays)
- Weekday Non-Rush
- Weekend

In [24]:
for alias in PLOT_STATIONS:
    df = dfs[alias]
    station = get_station(alias)
    fig = get_plot("rush_hour_analysis")(df, title=station.display_name)
    fig.show()

### Fourier Transform Analysis
Frequency spectrum of bike count signal reveals periodic patterns:
- **Daily cycle** (1/24 Hz): Main commute pattern
- **12h cycle**: Morning + evening rush peaks
- **Weekly cycle** (1/168 Hz): Weekday vs weekend differences

In [25]:
for alias in PLOT_STATIONS:
    df = dfs[alias]
    station = get_station(alias)
    fig = get_plot("fourier_transform")(df, title=f"{station.display_name} - Bike Counts")
    fig.show()

## 4. Multi-Station Comparison Plots

These plots compare patterns across multiple cities.

In [26]:
# Use the already-loaded dfs dict for multi-station comparison
city_data = dfs

### Hourly Pattern Comparison

In [27]:
fig = get_plot("city_comparison_hourly")(city_data)
fig.show()

### Monthly Pattern Comparison (Normalized)

In [28]:
fig = get_plot("city_comparison_monthly")(city_data)
fig.show()

### Winter to Summer Ratio
Shows seasonal swing by hour:
- **100%** = Winter equals Summer
- **50%** = Summer has 2x bikes of Winter

In [29]:
fig = get_plot("winter_summer_ratio")(city_data)
fig.show()

## 6. Adding New Stations

To add a new station, edit `tuecycle/config/stations.py`:

```python
STATIONS["my_new_station"] = Station(
    alias="my_new_station",
    city="CityName",  # Must match weather file name
    counter_name="Full Counter Name from CSV",
    display_name="Friendly Display Name",
    color="#HexColor"
)
```

Then load data as usual:
```python
df = dm.get("my_new_station")
```

## 7. Cache Management

The cache is stored in the `cache/` directory as Parquet files.

In [30]:
# List cached files
import os
cache_dir = "cache"
if os.path.exists(cache_dir):
    cached_files = os.listdir(cache_dir)
    print(f"Cached files ({len(cached_files)}):")
    for f in sorted(cached_files):
        size_mb = os.path.getsize(os.path.join(cache_dir, f)) / 1024 / 1024
        print(f"  • {f} ({size_mb:.2f} MB)")
else:
    print("No cache directory yet")

Cached files (65):
  • freiburg_dreisam_2024-11-01_2025-10-31.parquet (0.10 MB)
  • freiburg_eschholz_2024-11-01_2025-10-31.parquet (0.10 MB)
  • freiburg_gueterbahn_2024-11-01_2025-10-31.parquet (0.10 MB)
  • freiburg_wiwili_2024-11-01_2025-10-31.parquet (0.11 MB)
  • heidelberg_ernst_walz_2020-01-01_2025-11-30.parquet (0.54 MB)
  • heidelberg_ernst_walz_2024-11-01_2025-10-31.parquet (0.09 MB)
  • heidelberg_gaisberg_2020-01-01_2025-11-30.parquet (0.59 MB)
  • heidelberg_gaisberg_2024-11-01_2025-10-31.parquet (0.09 MB)
  • heidelberg_kurfuersten_2020-01-01_2025-11-30.parquet (0.59 MB)
  • heidelberg_kurfuersten_2024-11-01_2025-10-31.parquet (0.10 MB)
  • heidelberg_liebermann_2020-01-01_2025-11-30.parquet (0.58 MB)
  • heidelberg_liebermann_2024-11-01_2025-10-31.parquet (0.10 MB)
  • heidelberg_mannheimer_2020-01-01_2025-11-30.parquet (0.58 MB)
  • heidelberg_mannheimer_2024-11-01_2025-10-31.parquet (0.10 MB)
  • heidelberg_ploeck_2020-01-01_2025-11-30.parquet (0.59 MB)
  • heidelberg

In [31]:
# To clear the cache (forces reload from CSVs):
# dm.clear_cache()

# To preload all stations:
# dm.preload_all()

# The dm object defines the date range of the loaded data!